# Import, config and load data

In [16]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import *

from darts import TimeSeries
from darts.dataprocessing.transformers import WindowTransformer, StaticCovariatesTransformer

from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score, brier_score_loss,
    confusion_matrix,
)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)

import os
# -------------------- CONFIG --------------------
DATA_FOLDER       = "./data"
FIXED_DATA_PATH   = construct_path(DATA_FOLDER, "fixed")
DATASET_PATH      = construct_path(DATA_FOLDER, "dataset")

TARGET_RAW_EVENT_CLASSIFIER        = "act_drone_strike_on_ua"          # count target
TARGET_EVENT_CLASSIFIER            = "act_drone_strike_on_ua_binary"   # binary target
TARGET_REGRESSOR = TARGET_RAW_EVENT_CLASSIFIER

# Forecasting setup
OUTPUT_CHUNK_LEN  = 7      # how many days ahead each model predicts in one shot
INPUT_LAGS        = 7      # how many past days of the target the model sees
MULTI_MODELS      = True   # direct strategy: one sub-model per horizon step

# CV / split
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.10, 0.20   # enforced by split_series_list
CV_STRIDE                       = 1                   # For final evaluation No weekly stride, new forecast everyday


available_threads = get_available_threads()
print(f'CPU count: {available_threads}')

RANDOM_STATE      = 42
np.random.seed(RANDOM_STATE)

infra_types = ['health','education','residential','energy']
# intentions = ['intent','unintent']
intentions_oi = ['intent']
all_targets = [f"act_drone_infra_ua_{x}_{y}" for y in intentions_oi for x in infra_types]


CPU count: 16


In [17]:
regions,master_timeseries,regions_activity = \
    load_data(data_path=FIXED_DATA_PATH,dataset_path=DATASET_PATH)

master_timeseries shape: (847, 1622)
regions: 25  -  ['cherkasy', 'chernihiv', 'chernivtsi', 'dnipropetrovsk', 'donetsk'] ...
start: 2022-09-28 00:00:00 end: 2025-01-21 00:00:00


In [18]:
# print([x for x in master_timeseries.columns.tolist() if 'infra' in x])
# print(all_targets)

In [19]:
# A dataframe per infra type
damage_class_dfs, global_weather_columns = get_engineered_features_damageclassifiers(master_timeseries=master_timeseries,data_path=FIXED_DATA_PATH,targets_cols=all_targets,regions=regions,regions_activity=regions_activity)

damage_classes = dict()
for i,target in enumerate(all_targets):
    damage_classes[target] = {"raw_df" : damage_class_dfs[i]}
    
print('event_classifier:')
# For the event classification
for_global_EVENT_CLASSIFIER, _ = get_engineered_features(
    master_timeseries=master_timeseries,
    data_path=FIXED_DATA_PATH,
    target_col=TARGET_EVENT_CLASSIFIER, 
    regions=regions,
    regions_activity=regions_activity,
    binarize_target=True,
    target_raw_col=TARGET_RAW_EVENT_CLASSIFIER
)


act_drone_infra_ua_health_intent
Amount of target =1: 48.000
Total rows: 16940.000
Target positive rate: 0.003
act_drone_infra_ua_education_intent
Amount of target =1: 69.000
Total rows: 16940.000
Target positive rate: 0.004
act_drone_infra_ua_residential_intent
Amount of target =1: 390.000
Total rows: 16940.000
Target positive rate: 0.023
act_drone_infra_ua_energy_intent
Amount of target =1: 123.000
Total rows: 16940.000
Target positive rate: 0.007
event_classifier:
Shape after feature engineering: (16940, 115)
Target positive rate: 0.144


In [20]:
# For event_classifier
holiday_cols_c, future_covariates_c, exclude_cols_c, past_covariates_c = split_future_and_past_cov(for_global_EVENT_CLASSIFIER,global_weather_columns,TARGET_EVENT_CLASSIFIER)

# For damage_classifiers
for key,value in damage_classes.items():
    holiday_cols_d, future_covariates_d, exclude_cols_d, past_covariates_d = split_future_and_past_cov(damage_classes[key]['raw_df'],global_weather_columns,f"{key}_binary")
    damage_classes[key]['split_future_and_past_cov'] = {
        'H': holiday_cols_d, 'F': future_covariates_d, 'E': exclude_cols_d, 'P': past_covariates_d
    }

# future covariates : 27
# past covariates   : 85
# future covariates : 27
# past covariates   : 85
# future covariates : 27
# past covariates   : 85
# future covariates : 27
# past covariates   : 85
# future covariates : 27
# past covariates   : 85


### Building timeseries objects

In [21]:
# For event_classifier
target_series_list_c, past_covs_list_c,future_covs_list_c =build_ts_and_apply_window_transformer(
                                            for_global_EVENT_CLASSIFIER,
                                            TARGET_EVENT_CLASSIFIER,
                                            past_covariates_c,
                                            future_covariates_c,
                                            ed_alpha=halflife_to_alpha(7)
                                            )
# For damage classifiers
for key,value in damage_classes.items():
    target_series_list_d, past_covs_list_d,future_covs_list_d =build_ts_and_apply_window_transformer(
        for_global_reset=damage_classes[key]['raw_df'],
        target=f"{key}_binary",
        past_covariates=damage_classes[key]['split_future_and_past_cov']['P'],
        future_covariates=damage_classes[key]['split_future_and_past_cov']['F'],
        ed_alpha=halflife_to_alpha(7)
        )
    
    damage_classes[key]['build_ts_and_apply_window_transformer'] = {
        'T': target_series_list_d, 'P': past_covs_list_d, 'F': future_covs_list_d, 
    }

Past-cov components: raw=85  transformed=595
Past-cov components: raw=85  transformed=595
Past-cov components: raw=85  transformed=595
Past-cov components: raw=85  transformed=595
Past-cov components: raw=85  transformed=595


In [22]:
# For event_classifier
region_names, train_target_c, val_target_c, test_target_c, full_past_covs_c, full_fut_covs_c, target_for_cv_c, TRAIN_VAL_END,CV_START_VAL =\
      get_covs_and_encodings(target_series_list_c,past_covs_list_c,future_covs_list_c,TRAIN_FRAC,VAL_FRAC)

# For damage classifiers
for key,value in damage_classes.items():
    region_names, train_target_c, val_target_c, test_target_c, full_past_covs_c, full_fut_covs_c, target_for_cv_c, TRAIN_VAL_END,CV_START_VAL =\
      get_covs_and_encodings(
          damage_classes[key]['build_ts_and_apply_window_transformer']['T'],
          damage_classes[key]['build_ts_and_apply_window_transformer']['P'],
          damage_classes[key]['build_ts_and_apply_window_transformer']['F'],
          TRAIN_FRAC,VAL_FRAC
          )
    
    damage_classes[key]['get_covs_and_encodings'] = {
      'R': region_names, 
      'Tr': train_target_c,
      'V': val_target_c, 
      'Te':test_target_c, 
      'FPC':full_past_covs_c, 
      'FFC': full_fut_covs_c, 
      'TCV': target_for_cv_c, 
      'tve': TRAIN_VAL_END,
      'csv': CV_START_VAL
    }

# regions: 20
train len (region 0): 593 days
val   len (region 0): 85 days
test  len (region 0): 169 days  <- untouched until the final evaluation
CV view length       : 676 days (train+val)
CV start on that view: 0.875  -> rolls across the val segment only
# regions: 20
train len (region 0): 593 days
val   len (region 0): 85 days
test  len (region 0): 169 days  <- untouched until the final evaluation
CV view length       : 676 days (train+val)
CV start on that view: 0.875  -> rolls across the val segment only
# regions: 20
train len (region 0): 593 days
val   len (region 0): 85 days
test  len (region 0): 169 days  <- untouched until the final evaluation
CV view length       : 676 days (train+val)
CV start on that view: 0.875  -> rolls across the val segment only
# regions: 20
train len (region 0): 593 days
val   len (region 0): 85 days
test  len (region 0): 169 days  <- untouched until the final evaluation
CV view length       : 676 days (train+val)
CV start on that view: 0.875  -> ro

### Building the classifier and regressors themselves

In [23]:
from darts.models import LightGBMModel,LightGBMClassifierModel,SKLearnClassifierModel,CatBoostModel
from imbens.ensemble import SelfPacedEnsembleClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

COMMON_KWARGS = get_common_kwargs()

def get_event_classifier(**kwargs):
    DTm_depth = kwargs.get('DTm_depth', 5)
    SPE_estm  = kwargs.get('SPE_estm', 100)
    clf = SelfPacedEnsembleClassifier(
        estimator    = DecisionTreeClassifier(max_depth=DTm_depth, random_state=RANDOM_STATE),
        n_estimators = SPE_estm,
        random_state = RANDOM_STATE,
        n_jobs       = available_threads,
    )
    return SKLearnClassifierModel(model=clf, **COMMON_KWARGS)

def get_damage_classifier(**kwargs):
    DTm_depth = kwargs.get('DTm_depth', 5)
    SPE_estm  = kwargs.get('SPE_estm', 100)
    clf = SelfPacedEnsembleClassifier(
        estimator    = DecisionTreeClassifier(max_depth=DTm_depth, random_state=RANDOM_STATE),
        n_estimators = SPE_estm,
        random_state = RANDOM_STATE,
        n_jobs       = available_threads,
    )
    return SKLearnClassifierModel(model=clf, **COMMON_KWARGS)

def get_feature_selector_classifier(**kwargs):
    return LightGBMClassifierModel(
        **COMMON_KWARGS,
        objective    = "binary",
        is_unbalance = True,
        random_state = RANDOM_STATE,
        **kwargs
    )


In [24]:
full_weights = make_positive_only_weights(target_series_list_c)

In [ ]:
# For classifier
event_classifier_feature_selection = get_feature_selector_classifier()
event_classifier_feature_selection.fit(
    series          = train_target_c,
    past_covariates = full_past_covs_c,
    future_covariates = full_fut_covs_c,
)
top_100_features_c = get_top_100_from_lgbm(event_classifier_feature_selection)
top_100_features_dict_c = clean_feature_names(top_100_features_c)

for key,value in damage_classes.items():
    damage_classifier_feature_selection = get_feature_selector_classifier()
    damage_classifier_feature_selection.fit(
        series          = damage_classes[key]['get_covs_and_encodings']['Tr'],
        past_covariates = damage_classes[key]['get_covs_and_encodings']['FPC'],
        future_covariates = damage_classes[key]['get_covs_and_encodings']['FFC'],
        sample_weight     = [w.slice_intersect(ts) for w, ts in zip(full_weights, damage_classes[key]['get_covs_and_encodings']['Tr'])],
        #Train only on samples where the event of drone striking happened
    )
    top_100_features_d = get_top_100_from_lgbm(damage_classifier_feature_selection)
    top_100_features_dict_d = clean_feature_names(top_100_features_d)
    damage_classes[key]['feature_selection']  = {'top_100_features_d': top_100_features_d, 'top_100_features_dict_d': top_100_features_dict_d}

[LightGBM] [Info] Number of positive: 35, number of negative: 11425
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.196841 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 300855
[LightGBM] [Info] Number of data points in the train set: 11460, number of used features: 2097
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.003054 -> initscore=-5.788211
[LightGBM] [Info] Start training from score -5.788211
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

In [26]:
# For classifier
past_covs_list_c   = [subset_safe(ts, top_100_features_dict_c['pastcov_features_base']) for ts in past_covs_list_c]
future_covs_list_c = [subset_safe(ts, top_100_features_dict_c['futcov_features_base'])  for ts in future_covs_list_c]
_, train_target_c, val_target_c, test_target_c, full_past_covs_c, full_fut_covs_c, target_for_cv_c,TRAIN_VAL_END,CV_START_VAL = \
    get_covs_and_encodings(target_series_list_c,past_covs_list_c,future_covs_list_c,TRAIN_FRAC,VAL_FRAC)

for key,value in damage_classes.items():
    b1 = damage_classes[key]['feature_selection']['top_100_features_dict_d']
    b2 = damage_classes[key]['build_ts_and_apply_window_transformer']
    past_covs_list_d   = [subset_safe(ts, b1['pastcov_features_base']) for ts in b2['P']]
    future_covs_list_d = [subset_safe(ts, b1['futcov_features_base'])  for ts in b2['F']]
    _, train_target_d, val_target_d, test_target_d, full_past_covs_d, full_fut_covs_d, target_for_cv_d,TRAIN_VAL_END,CV_START_VAL = \
        get_covs_and_encodings(b2['T'],past_covs_list_d,future_covs_list_d,TRAIN_FRAC,VAL_FRAC)
    
    damage_classes[key]['get_covs_and_encodings'] = {
      'R': region_names, 
      'Tr': train_target_d,
      'V': val_target_d, 
      'Te':test_target_d, 
      'FPC':full_past_covs_d, 
      'FFC': full_fut_covs_d, 
      'TCV': target_for_cv_d, 
      'tve': TRAIN_VAL_END,
      'csv': CV_START_VAL
    }

# regions: 20
train len (region 0): 593 days
val   len (region 0): 85 days
test  len (region 0): 169 days  <- untouched until the final evaluation
CV view length       : 676 days (train+val)
CV start on that view: 0.875  -> rolls across the val segment only
# regions: 20
train len (region 0): 593 days
val   len (region 0): 85 days
test  len (region 0): 169 days  <- untouched until the final evaluation
CV view length       : 676 days (train+val)
CV start on that view: 0.875  -> rolls across the val segment only
# regions: 20
train len (region 0): 593 days
val   len (region 0): 85 days
test  len (region 0): 169 days  <- untouched until the final evaluation
CV view length       : 676 days (train+val)
CV start on that view: 0.875  -> rolls across the val segment only
# regions: 20
train len (region 0): 593 days
val   len (region 0): 85 days
test  len (region 0): 169 days  <- untouched until the final evaluation
CV view length       : 676 days (train+val)
CV start on that view: 0.875  -> ro

# Predictions
- Probability that the drone strike event happens $\hat{\pi}(x) = P(Y > 0 | X)$

- Probability that a certain type of infrastructure gets damaged **given** the strike $\hat{\delta}(x) = P(X | Y > 0, D)$

- Final estimate $\hat{P}(X|D) = \hat{\pi}(x) \cdot \hat{\delta}(x)$


## Cross-Validation

In [27]:
def _classif_metrics_hurdle(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int).ravel()
    y_prob = np.asarray(y_prob).astype(float).ravel()
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    def _safe(fn_):
        try: return float(fn_())
        except (ValueError, ZeroDivisionError): return np.nan
    return {
        "F1":        _safe(lambda: f1_score(y_true, y_pred, zero_division=0)),
        "Precision": _safe(lambda: precision_score(y_true, y_pred, zero_division=0)),
        "Recall":    _safe(lambda: recall_score(y_true, y_pred, zero_division=0)),
        "ROC_AUC":   _safe(lambda: roc_auc_score(y_true, y_prob)),
        "PR_AUC":    _safe(lambda: average_precision_score(y_true, y_prob)),
        "Brier":     _safe(lambda: brier_score_loss(y_true, y_prob)),
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn), "n": int(len(y_true)),
    }


def evaluate_classif_long(long_df, threshold=0.5):
    def _rows(group_cols):
        rows = []
        for keys, sub in long_df.groupby(group_cols):
            m = _classif_metrics_hurdle(sub["y_true"], sub["y_prob"], threshold)
            key_map = (
                {group_cols: keys} if isinstance(group_cols, str)
                else dict(zip(group_cols, keys if isinstance(keys, tuple) else (keys,)))
            )
            m.update(key_map)
            rows.append(m)
        return pd.DataFrame(rows)
    return {
        "per_region":         _rows("region").sort_values("F1", ascending=False).reset_index(drop=True),
        "per_horizon":        _rows("horizon").sort_values("horizon").reset_index(drop=True),
        "per_region_horizon": _rows(["region", "horizon"]).sort_values(["region", "horizon"]).reset_index(drop=True),
        "global":             _classif_metrics_hurdle(long_df["y_true"], long_df["y_prob"], threshold),
    }


In [28]:
def run_hurdle_cv(predict_stride=1, retrain_stride=OUTPUT_CHUNK_LEN):
    """Expanding-window CV for the hurdle classifier.

    Stage 1: event classifier  → P(drone strike)
    Stage 2: damage classifiers (one per infra type) → P(damage | strike)
    Hurdle:  P(damage) = P(strike) * P(damage | strike)

    Returns
    -------
    fold_preds_c : list[list[TimeSeries]]               P(event) per region
    fold_preds_d : dict[key → list[list[TimeSeries]]]   P(damage|event) per type, per region
    fold_preds_h : dict[key → list[list[TimeSeries]]]   hurdle prob per type, per region
    """
    ref_ts    = target_for_cv_c[0]
    n_total   = len(ref_ts)
    start_idx = int(CV_START_VAL * n_total)
    n_regions = len(target_for_cv_c)

    fold_preds_c = [[] for _ in range(n_regions)]
    fold_preds_d = {key: [[] for _ in range(n_regions)] for key in damage_classes}
    fold_preds_h = {key: [[] for _ in range(n_regions)] for key in damage_classes}
    n_preds    = 0
    n_retrains = 0
    clf = None
    damage_clfs = {key: None for key in damage_classes}

    for t0 in range(start_idx, n_total - OUTPUT_CHUNK_LEN + 1, predict_stride):
        steps_since_start = t0 - start_idx

        if steps_since_start % retrain_stride == 0:
            retrain_time  = ref_ts.time_index[t0]
            train_c       = [ts.drop_after(retrain_time) for ts in target_for_cv_c]
            train_weights = [w.drop_after(retrain_time)  for w  in full_weights]

            clf = get_event_classifier()
            clf.fit(series=train_c, past_covariates=full_past_covs_c,
                    future_covariates=full_fut_covs_c)

            for key in damage_classes:
                dc = damage_classes[key]
                target_for_cv_d = dc['get_covs_and_encodings']['TCV']
                full_past_covs_d = dc['get_covs_and_encodings']['FPC']
                full_fut_covs_d  = dc['get_covs_and_encodings']['FFC']
                train_d = [ts.drop_after(retrain_time) for ts in target_for_cv_d]
                dmg_clf = get_damage_classifier()
                dmg_clf.fit(
                    series=train_d,
                    past_covariates=full_past_covs_d,
                    future_covariates=full_fut_covs_d,
                    sample_weight=[w.slice_intersect(ts) for w, ts in zip(train_weights, train_d)],
                )
                damage_clfs[key] = dmg_clf

            n_retrains += 1
            print(f"   retrain {n_retrains}  (data up to {retrain_time.date()})")

        split_time    = ref_ts.time_index[t0]
        pred_series_c = [ts.drop_after(split_time) for ts in target_for_cv_c]

        preds_c = clf.predict(
            n=OUTPUT_CHUNK_LEN, series=pred_series_c,
            past_covariates=full_past_covs_c, future_covariates=full_fut_covs_c,
            predict_likelihood_parameters=True, show_warnings=False,
        )

        preds_d_all = {}
        for key in damage_classes:
            dc = damage_classes[key]
            target_for_cv_d = dc['get_covs_and_encodings']['TCV']
            full_past_covs_d = dc['get_covs_and_encodings']['FPC']
            full_fut_covs_d  = dc['get_covs_and_encodings']['FFC']
            pred_series_d = [ts.drop_after(split_time) for ts in target_for_cv_d]
            preds_d_all[key] = damage_clfs[key].predict(
                n=OUTPUT_CHUNK_LEN, series=pred_series_d,
                past_covariates=full_past_covs_d, future_covariates=full_fut_covs_d,
                predict_likelihood_parameters=True, show_warnings=False,
            )

        for r_idx, pred_c in enumerate(preds_c):
            prob_event_ts = pred_c.univariate_component(pred_c.n_components - 1)
            prob_event    = prob_event_ts.values().ravel()
            fold_preds_c[r_idx].append(prob_event_ts)

            for key in damage_classes:
                pred_d     = preds_d_all[key][r_idx]
                prob_dmg_ts = pred_d.univariate_component(pred_d.n_components - 1)
                prob_dmg    = prob_dmg_ts.values().ravel()
                hurdle_ts   = TimeSeries.from_times_and_values(
                    prob_event_ts.time_index, (prob_event * prob_dmg).reshape(-1, 1)
                )
                fold_preds_d[key][r_idx].append(prob_dmg_ts)
                fold_preds_h[key][r_idx].append(hurdle_ts)

        n_preds += 1

    print(f"   {n_preds} daily predictions, {n_retrains} retrains complete")
    return fold_preds_c, fold_preds_d, fold_preds_h


In [29]:
print("Running hurdle CV ...")
fold_preds_c, fold_preds_d, fold_preds_h = run_hurdle_cv()

# Event classifier
long_df_c = (
    collect_predictions_long(target_for_cv_c, fold_preds_c, region_names)
    .rename(columns={"y_pred": "y_prob"})
)
print(f"Event classifier rows : {len(long_df_c):,}")

# Damage classifiers and hurdle — one long_df per infra type
long_df_d = {}
long_df_h = {}
for key in damage_classes:
    target_for_cv_d = damage_classes[key]['get_covs_and_encodings']['TCV']
    long_df_d[key] = (
        collect_predictions_long(target_for_cv_d, fold_preds_d[key], region_names)
        .rename(columns={"y_pred": "y_prob"})
    )
    long_df_h[key] = (
        collect_predictions_long(target_for_cv_d, fold_preds_h[key], region_names)
        .rename(columns={"y_pred": "y_prob"})
    )
    print(f"  [{key}]  damage rows: {len(long_df_d[key]):,}  hurdle rows: {len(long_df_h[key]):,}")


This model will treat target `series` lagged values as numeric input features (and not categorical).


Running hurdle CV ...


This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).


   retrain 1  (data up to 2024-05-11)


This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).


...

   retrain 10  (data up to 2024-07-13)


This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).


   retrain 11  (data up to 2024-07-20)


This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).


   retrain 12  (data up to 2024-07-27)
   79 daily predictions, 12 retrains complete
Event classifier rows : 11,060
  [act_drone_infra_ua_health_intent]  damage rows: 11,060  hurdle rows: 11,060
  [act_drone_infra_ua_education_intent]  damage rows: 11,060  hurdle rows: 11,060
  [act_drone_infra_ua_residential_intent]  damage rows: 11,060  hurdle rows: 11,060
  [act_drone_infra_ua_energy_intent]  damage rows: 11,060  hurdle rows: 11,060


In [30]:
cv_classif_results_c = evaluate_classif_long(long_df_c)
gc = cv_classif_results_c["global"]
print("=== Event Classifier CV — Global metrics ===")
print(f"  F1      : {gc['F1']:.4f}")
print(f"  ROC-AUC : {gc['ROC_AUC']:.4f}")
print(f"  PR-AUC  : {gc['PR_AUC']:.4f}")
print(f"  Brier   : {gc['Brier']:.4f}")
print(f"  n       : {gc['n']:,}")

cv_classif_results_d = {}
cv_classif_results_h = {}
for key in damage_classes:
    cv_classif_results_d[key] = evaluate_classif_long(long_df_d[key])
    cv_classif_results_h[key] = evaluate_classif_long(long_df_h[key])
    gd = cv_classif_results_d[key]["global"]
    gh = cv_classif_results_h[key]["global"]
    label = key.replace("act_drone_infra_ua_", "").replace("_intent", "")
    print(f"\n=== [{label}] Damage Classifier CV — Global ===")
    print(f"  F1      : {gd['F1']:.4f}  |  ROC-AUC : {gd['ROC_AUC']:.4f}  |  PR-AUC : {gd['PR_AUC']:.4f}  |  Brier : {gd['Brier']:.4f}")
    print(f"=== [{label}] Hurdle CV — Global ===")
    print(f"  F1      : {gh['F1']:.4f}  |  ROC-AUC : {gh['ROC_AUC']:.4f}  |  PR-AUC : {gh['PR_AUC']:.4f}  |  Brier : {gh['Brier']:.4f}")


=== Event Classifier CV — Global metrics ===
  F1      : 0.6488
  ROC-AUC : 0.9016
  PR-AUC  : 0.7241
  Brier   : 0.1493
  n       : 11,060

=== [health] Damage Classifier CV — Global ===
  F1      : 0.0065  |  ROC-AUC : 0.4652  |  PR-AUC : 0.0032  |  Brier : 0.6940
=== [health] Hurdle CV — Global ===
  F1      : 0.0207  |  ROC-AUC : 0.8211  |  PR-AUC : 0.0104  |  Brier : 0.1387

=== [education] Damage Classifier CV — Global ===
  F1      : 0.0080  |  ROC-AUC : 0.3668  |  PR-AUC : 0.0032  |  Brier : 0.7425
=== [education] Hurdle CV — Global ===
  F1      : 0.0235  |  ROC-AUC : 0.6884  |  PR-AUC : 0.0253  |  Brier : 0.1440

=== [residential] Damage Classifier CV — Global ===
  F1      : 0.0563  |  ROC-AUC : 0.4073  |  PR-AUC : 0.0239  |  Brier : 0.7434
=== [residential] Hurdle CV — Global ===
  F1      : 0.1518  |  ROC-AUC : 0.8200  |  PR-AUC : 0.1175  |  Brier : 0.1478

=== [energy] Damage Classifier CV — Global ===
  F1      : 0.0291  |  ROC-AUC : 0.5141  |  PR-AUC : 0.0157  |  Brier 

In [31]:
print("=== Event Classifier CV — Per-horizon ===")
print(cv_classif_results_c["per_horizon"][["horizon", "F1", "ROC_AUC", "PR_AUC", "Brier", "n"]])

for key in damage_classes:
    label = key.replace("act_drone_infra_ua_", "").replace("_intent", "")
    print(f"\n=== [{label}] Damage Classifier CV — Per-horizon ===")
    print(cv_classif_results_d[key]["per_horizon"][["horizon", "F1", "ROC_AUC", "PR_AUC", "Brier", "n"]])
    print(f"=== [{label}] Hurdle CV — Per-horizon ===")
    print(cv_classif_results_h[key]["per_horizon"][["horizon", "F1", "ROC_AUC", "PR_AUC", "Brier", "n"]])


=== Event Classifier CV — Per-horizon ===


,horizon,F1,ROC_AUC,PR_AUC,Brier,n
0,1,0.649880,0.898900,0.723221,0.150116,1580
1,2,0.645933,0.902669,0.730915,0.146665,1580
2,3,0.640187,0.901367,0.706678,0.150813,1580
3,4,0.653159,0.904728,0.720032,0.149248,1580
4,5,0.648585,0.899657,0.723588,0.149058,1580
5,6,0.647887,0.901876,0.747019,0.149020,1580
6,7,0.656325,0.902385,0.726877,0.150106,1580



=== [health] Damage Classifier CV — Per-horizon ===


,horizon,F1,ROC_AUC,PR_AUC,Brier,n
0,1,0.006349,0.558667,0.004570,0.667157,1580
1,2,0.006309,0.527937,0.005384,0.756760,1580
2,3,0.005131,0.354413,0.002718,0.681349,1580
3,4,0.006329,0.436444,0.003041,0.754106,1580
4,5,0.006341,0.492376,0.008023,0.641854,1580
5,6,0.007595,0.414602,0.003765,0.635042,1580
6,7,0.007638,0.494653,0.004891,0.721424,1580


=== [health] Hurdle CV — Per-horizon ===


...

,horizon,F1,ROC_AUC,PR_AUC,Brier,n
0,1,0.163793,0.828587,0.132904,0.146947,1580
1,2,0.138393,0.799642,0.127263,0.146825,1580
2,3,0.137339,0.810541,0.103804,0.152531,1580
3,4,0.152381,0.827844,0.126269,0.143762,1580
4,5,0.144000,0.818919,0.114459,0.153734,1580
5,6,0.169565,0.854683,0.198538,0.147056,1580
6,7,0.158140,0.805069,0.110125,0.143488,1580



=== [energy] Damage Classifier CV — Per-horizon ===


,horizon,F1,ROC_AUC,PR_AUC,Brier,n
0,1,0.026261,0.456459,0.014639,0.769655,1580
1,2,0.026352,0.524408,0.017732,0.725180,1580
2,3,0.031270,0.563143,0.017826,0.762292,1580
3,4,0.030144,0.528492,0.016284,0.760444,1580
4,5,0.029487,0.579745,0.021719,0.805543,1580
5,6,0.031289,0.432849,0.014701,0.829697,1580
6,7,0.028728,0.503682,0.017180,0.741136,1580


=== [energy] Hurdle CV — Per-horizon ===


,horizon,F1,ROC_AUC,PR_AUC,Brier,n
0,1,0.053812,0.583146,0.069630,0.157124,1580
1,2,0.058394,0.603470,0.108543,0.145854,1580
2,3,0.060345,0.691838,0.043471,0.158773,1580
3,4,0.058166,0.686241,0.049464,0.156123,1580
4,5,0.055675,0.670282,0.057846,0.162050,1580
5,6,0.062622,0.673646,0.041515,0.168166,1580
6,7,0.056075,0.612870,0.035843,0.151708,1580


## Classifier Calibration

In [32]:
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

_CAL_EPS = 1e-6

def _logit(p):
    p = np.clip(p, _CAL_EPS, 1 - _CAL_EPS)
    return np.log(p / (1 - p))

def fit_sigmoid_cal(y_prob, y_true):
    lr = LogisticRegression(C=1e6, solver="lbfgs")
    lr.fit(_logit(y_prob).reshape(-1, 1), np.asarray(y_true, dtype=int))
    return lr

def apply_sigmoid_cal(lr, y_prob):
    return lr.predict_proba(_logit(y_prob).reshape(-1, 1))[:, 1]

def _oof_one_group(y, p, method, n_splits, random_state):
    """OOF calibrated probs for one contiguous subset."""
    minority = int(min(np.bincount(y))) if set(y) == {0, 1} else 0
    n_splits_eff = max(2, min(n_splits, minority)) if minority >= 2 else 0
    if n_splits_eff < 2:
        return apply_sigmoid_cal(fit_sigmoid_cal(p, y), p)
    out = np.zeros_like(p)
    skf = StratifiedKFold(n_splits=n_splits_eff, shuffle=True, random_state=random_state)
    for fit_idx, eval_idx in skf.split(p, y):
            c = fit_sigmoid_cal(p[fit_idx], y[fit_idx])
            out[eval_idx] = apply_sigmoid_cal(c, p[eval_idx])
    return out


def oof_calibrated_probs(long_df, method, n_splits=5, random_state=RANDOM_STATE, group_col=None):
    """Returns (y_true, p_raw, p_calibrated) arrays aligned to long_df rows."""
    y = long_df["y_true"].to_numpy().astype(int)
    p = long_df["y_prob"].to_numpy().astype(float)
    if group_col is None:
        return y, p, _oof_one_group(y, p, method, n_splits, random_state)
    p_cal  = np.zeros_like(p)
    groups = long_df[group_col].to_numpy()
    for g in pd.unique(groups):
        mask = (groups == g)
        p_cal[mask] = _oof_one_group(y[mask], p[mask], method, n_splits, random_state)
    return y, p, p_cal


def fit_final_calibrators_per_horizon(long_df, method):
    """Fit one calibrator per horizon on all CV rows. Returns {horizon: calibrator}."""
    out = {}
    for h, sub in long_df.groupby("horizon"):
        y_h = sub["y_true"].to_numpy().astype(int)
        p_h = sub["y_prob"].to_numpy().astype(float)
        if len(np.unique(y_h)) < 2:
            out[int(h)] = None
            continue
        out[int(h)] = fit_sigmoid_cal(p_h, y_h)
    return out


def apply_calibrator_per_horizon(long_df, calibrators_per_h, method):
    """Apply per-horizon calibrator to y_prob column. Returns calibrated prob array."""
    p_raw = long_df["y_prob"].to_numpy().astype(float)
    out   = p_raw.copy()
    for h, sub in long_df.groupby("horizon"):
        cal = calibrators_per_h.get(int(h))
        if cal is None:
            continue
        idx   = sub.index.to_numpy()
        sub_p = sub["y_prob"].to_numpy().astype(float)
        out[idx] =apply_sigmoid_cal(cal, sub_p)
    return out


In [33]:
# OOF calibration assessment — compare Brier before vs after for event + each damage type
long_df_c_reset = long_df_c.reset_index(drop=True)

print("=== Event classifier ===")
for method in ["sigmoid"]:
    y_true_cal, y_raw, y_cal = oof_calibrated_probs(long_df_c_reset, method=method, group_col="horizon")
    g_raw = _classif_metrics_hurdle(y_true_cal, y_raw)
    g_cal = _classif_metrics_hurdle(y_true_cal, y_cal)
    print(f"  {method:10s}  Brier {g_raw['Brier']:.4f} -> {g_cal['Brier']:.4f} "
          f"| ROC-AUC {g_raw['ROC_AUC']:.4f} -> {g_cal['ROC_AUC']:.4f} "
          f"| PR-AUC {g_raw['PR_AUC']:.4f} -> {g_cal['PR_AUC']:.4f}")

for key in damage_classes:
    label = key.replace("act_drone_infra_ua_", "").replace("_intent", "")
    print(f"\n=== Damage classifier [{label}] ===")
    df_reset = long_df_d[key].reset_index(drop=True)
    for method in ["sigmoid"]:
        y_true_cal, y_raw, y_cal = oof_calibrated_probs(df_reset, method=method, group_col="horizon")
        g_raw = _classif_metrics_hurdle(y_true_cal, y_raw)
        g_cal = _classif_metrics_hurdle(y_true_cal, y_cal)
        print(f"  {method:10s}  Brier {g_raw['Brier']:.4f} -> {g_cal['Brier']:.4f} "
              f"| ROC-AUC {g_raw['ROC_AUC']:.4f} -> {g_cal['ROC_AUC']:.4f} "
              f"| PR-AUC {g_raw['PR_AUC']:.4f} -> {g_cal['PR_AUC']:.4f}")


=== Event classifier ===
  sigmoid     Brier 0.1493 -> 0.0902 | ROC-AUC 0.9016 -> 0.9006 | PR-AUC 0.7241 -> 0.7217

=== Damage classifier [health] ===
  sigmoid     Brier 0.6940 -> 0.0034 | ROC-AUC 0.4652 -> 0.3639 | PR-AUC 0.0032 -> 0.0026

=== Damage classifier [education] ===
  sigmoid     Brier 0.7425 -> 0.0043 | ROC-AUC 0.3668 -> 0.5915 | PR-AUC 0.0032 -> 0.0064

=== Damage classifier [residential] ===
  sigmoid     Brier 0.7434 -> 0.0290 | ROC-AUC 0.4073 -> 0.5822 | PR-AUC 0.0239 -> 0.0402

=== Damage classifier [energy] ===
  sigmoid     Brier 0.7706 -> 0.0151 | ROC-AUC 0.5141 -> 0.4888 | PR-AUC 0.0157 -> 0.0165


In [34]:
CAL_METHOD = "sigmoid"

# --- Event classifier calibrators ---
calibrators_event = fit_final_calibrators_per_horizon(long_df_c_reset, method=CAL_METHOD)
print(f"Event: fitted {sum(c is not None for c in calibrators_event.values())}/{len(calibrators_event)} "
      f"per-horizon calibrators  (method={CAL_METHOD!r})")

long_df_c_cal           = long_df_c_reset.copy()
long_df_c_cal["y_prob"] = apply_calibrator_per_horizon(long_df_c_reset, calibrators_event, method=CAL_METHOD)

gc_raw = cv_classif_results_c["global"]
gc_cal = evaluate_classif_long(long_df_c_cal)["global"]
print(f"\nEvent classifier after {CAL_METHOD} calibration:")
print(f"  Brier   : {gc_raw['Brier']:.4f}  ->  {gc_cal['Brier']:.4f}")
print(f"  ROC-AUC : {gc_raw['ROC_AUC']:.4f}  ->  {gc_cal['ROC_AUC']:.4f}")
print(f"  PR-AUC  : {gc_raw['PR_AUC']:.4f}  ->  {gc_cal['PR_AUC']:.4f}")

# --- Damage classifier calibrators (one set per infra type) ---
damage_calibrators  = {}
long_df_d_cal       = {}
long_df_h_cal       = {}

for key in damage_classes:
    label = key.replace("act_drone_infra_ua_", "").replace("_intent", "")
    df_d_reset = long_df_d[key].reset_index(drop=True)
    damage_calibrators[key] = fit_final_calibrators_per_horizon(df_d_reset, method=CAL_METHOD)
    print(f"\n[{label}] damage: fitted "
          f"{sum(c is not None for c in damage_calibrators[key].values())}/{len(damage_calibrators[key])} calibrators")

    # Apply calibration to damage predictions
    long_df_d_cal[key]           = df_d_reset.copy()
    long_df_d_cal[key]["y_prob"] = apply_calibrator_per_horizon(df_d_reset, damage_calibrators[key], method=CAL_METHOD)

    # Reconstruct hurdle with calibrated event * calibrated damage
    df_h_reset = long_df_h[key].reset_index(drop=True).copy()
    df_h_reset["y_prob"] = long_df_c_cal["y_prob"].values * long_df_d_cal[key]["y_prob"].values
    long_df_h_cal[key] = df_h_reset

    gd_raw = cv_classif_results_d[key]["global"]
    gd_cal = evaluate_classif_long(long_df_d_cal[key])["global"]
    gh_raw = cv_classif_results_h[key]["global"]
    gh_cal = evaluate_classif_long(long_df_h_cal[key])["global"]

    print(f"  Damage  Brier {gd_raw['Brier']:.4f} -> {gd_cal['Brier']:.4f} "
          f"| ROC-AUC {gd_raw['ROC_AUC']:.4f} -> {gd_cal['ROC_AUC']:.4f} "
          f"| PR-AUC {gd_raw['PR_AUC']:.4f} -> {gd_cal['PR_AUC']:.4f}")
    print(f"  Hurdle  Brier {gh_raw['Brier']:.4f} -> {gh_cal['Brier']:.4f} "
          f"| ROC-AUC {gh_raw['ROC_AUC']:.4f} -> {gh_cal['ROC_AUC']:.4f} "
          f"| PR-AUC {gh_raw['PR_AUC']:.4f} -> {gh_cal['PR_AUC']:.4f}")

print("\n=== Calibrated Event Classifier — Per-horizon ===")
print(evaluate_classif_long(long_df_c_cal)["per_horizon"][["horizon", "F1", "ROC_AUC", "PR_AUC", "Brier", "n"]])

for key in damage_classes:
    label = key.replace("act_drone_infra_ua_", "").replace("_intent", "")
    print(f"\n=== [{label}] Calibrated Hurdle — Per-horizon ===")
    print(evaluate_classif_long(long_df_h_cal[key])["per_horizon"][["horizon", "F1", "ROC_AUC", "PR_AUC", "Brier", "n"]])


Event: fitted 7/7 per-horizon calibrators  (method='sigmoid')

Event classifier after sigmoid calibration:
  Brier   : 0.1493  ->  0.0900
  ROC-AUC : 0.9016  ->  0.9017
  PR-AUC  : 0.7241  ->  0.7244

[health] damage: fitted 7/7 calibrators
  Damage  Brier 0.6940 -> 0.0034 | ROC-AUC 0.4652 -> 0.5479 | PR-AUC 0.0032 -> 0.0059
  Hurdle  Brier 0.1387 -> 0.0034 | ROC-AUC 0.8211 -> 0.8577 | PR-AUC 0.0104 -> 0.0154

[education] damage: fitted 7/7 calibrators
  Damage  Brier 0.7425 -> 0.0043 | ROC-AUC 0.3668 -> 0.6588 | PR-AUC 0.0032 -> 0.0090
  Hurdle  Brier 0.1440 -> 0.0043 | ROC-AUC 0.6884 -> 0.7967 | PR-AUC 0.0253 -> 0.0175

[residential] damage: fitted 7/7 calibrators
  Damage  Brier 0.7434 -> 0.0289 | ROC-AUC 0.4073 -> 0.5923 | PR-AUC 0.0239 -> 0.0446
  Hurdle  Brier 0.1478 -> 0.0289 | ROC-AUC 0.8200 -> 0.8692 | PR-AUC 0.1175 -> 0.1344

[energy] damage: fitted 7/7 calibrators
  Damage  Brier 0.7706 -> 0.0150 | ROC-AUC 0.5141 -> 0.5381 | PR-AUC 0.0157 -> 0.0198
  Hurdle  Brier 0.1571 -> 

,horizon,F1,ROC_AUC,PR_AUC,Brier,n
0,1,0.663265,0.898900,0.723221,0.089268,1580
1,2,0.666667,0.902669,0.730915,0.088373,1580
2,3,0.674457,0.901367,0.706678,0.090190,1580
3,4,0.662316,0.904728,0.720032,0.089426,1580
4,5,0.664474,0.899657,0.723588,0.091524,1580
5,6,0.673203,0.901876,0.747019,0.090572,1580
6,7,0.665605,0.902385,0.726877,0.090794,1580



=== [health] Calibrated Hurdle — Per-horizon ===


,horizon,F1,ROC_AUC,PR_AUC,Brier,n
0,1,0.0,0.899683,0.024666,0.003153,1580
1,2,0.0,0.900698,0.020908,0.003153,1580
2,3,0.0,0.895873,0.023913,0.003151,1580
3,4,0.0,0.869587,0.013610,0.003154,1580
4,5,0.0,0.814380,0.027332,0.003784,1580
5,6,0.0,0.848899,0.022665,0.003781,1580
6,7,0.0,0.809509,0.015660,0.003785,1580



=== [education] Calibrated Hurdle — Per-horizon ===


,horizon,F1,ROC_AUC,PR_AUC,Brier,n
0,1,0.0,0.764189,0.028685,0.003784,1580
1,2,0.0,0.821156,0.023211,0.003777,1580
2,3,0.0,0.807497,0.027013,0.003777,1580
3,4,0.0,0.806375,0.024493,0.004411,1580
4,5,0.0,0.800018,0.019724,0.004407,1580
5,6,0.0,0.801209,0.057325,0.005023,1580
6,7,0.0,0.767096,0.027884,0.005038,1580



=== [residential] Calibrated Hurdle — Per-horizon ===


,horizon,F1,ROC_AUC,PR_AUC,Brier,n
0,1,0.0,0.866718,0.136765,0.029278,1580
1,2,0.0,0.862958,0.146606,0.028693,1580
2,3,0.0,0.877870,0.164172,0.027975,1580
3,4,0.0,0.877934,0.132289,0.028736,1580
4,5,0.0,0.867412,0.180217,0.029327,1580
5,6,0.0,0.873803,0.148962,0.029393,1580
6,7,0.0,0.859919,0.113892,0.028824,1580



=== [energy] Calibrated Hurdle — Per-horizon ===


,horizon,F1,ROC_AUC,PR_AUC,Brier,n
0,1,0.0,0.693927,0.046387,0.015003,1580
1,2,0.0,0.694462,0.041370,0.015022,1580
2,3,0.0,0.678717,0.048322,0.015023,1580
3,4,0.0,0.677619,0.045993,0.015021,1580
4,5,0.0,0.680136,0.053108,0.015021,1580
5,6,0.0,0.682341,0.053281,0.015626,1580
6,7,0.0,0.676467,0.046740,0.015023,1580


## Final Test Set Evaluation

In [35]:
def run_final_test(predict_stride=1, retrain_stride=OUTPUT_CHUNK_LEN):
    """Evaluate hurdle classifier on the held-out test set.

    Stage 1: event classifier  → P(drone strike)
    Stage 2: damage classifiers (one per infra type) → P(damage | strike)
    Hurdle:  P(damage) = P(strike) * P(damage | strike)

    Returns
    -------
    fold_preds_c  : list[list[TimeSeries]]              P(event) per region
    fold_preds_d  : dict[key → list[list[TimeSeries]]]  P(damage|event) per type, per region
    fold_preds_h  : dict[key → list[list[TimeSeries]]]  hurdle prob per type, per region
    full_target_c : list[TimeSeries]                    full event target (train+val+test)
    full_target_d : dict[key → list[TimeSeries]]        full damage targets per type
    """
    full_target_c = [tr.append(vl).append(te)
                     for tr, vl, te in zip(train_target_c, val_target_c, test_target_c)]
    full_target_d = {}
    for key in damage_classes:
        gc = damage_classes[key]['get_covs_and_encodings']
        full_target_d[key] = [tr.append(vl).append(te)
                               for tr, vl, te in zip(gc['Tr'], gc['V'], gc['Te'])]

    full_weights_final = make_positive_only_weights(full_target_c)

    ref_ts         = full_target_c[0]
    n_total        = len(ref_ts)
    test_start_idx = int(TRAIN_VAL_END * n_total)
    n_regions      = len(full_target_c)

    fold_preds_c = [[] for _ in range(n_regions)]
    fold_preds_d = {key: [[] for _ in range(n_regions)] for key in damage_classes}
    fold_preds_h = {key: [[] for _ in range(n_regions)] for key in damage_classes}
    n_preds    = 0
    n_retrains = 0
    clf = None
    damage_clfs = {key: None for key in damage_classes}

    for t0 in range(test_start_idx, n_total - OUTPUT_CHUNK_LEN + 1, predict_stride):
        steps_since_start = t0 - test_start_idx

        if steps_since_start % retrain_stride == 0:
            retrain_time  = ref_ts.time_index[t0]
            train_c       = [ts.drop_after(retrain_time) for ts in full_target_c]
            train_weights = [w.drop_after(retrain_time)  for w  in full_weights_final]

            clf = get_event_classifier()
            clf.fit(series=train_c, past_covariates=full_past_covs_c,
                    future_covariates=full_fut_covs_c)

            for key in damage_classes:
                dc = damage_classes[key]
                full_past_covs_d = dc['get_covs_and_encodings']['FPC']
                full_fut_covs_d  = dc['get_covs_and_encodings']['FFC']
                train_d = [ts.drop_after(retrain_time) for ts in full_target_d[key]]
                dmg_clf = get_damage_classifier()
                dmg_clf.fit(
                    series=train_d,
                    past_covariates=full_past_covs_d,
                    future_covariates=full_fut_covs_d,
                    sample_weight=[w.slice_intersect(ts) for w, ts in zip(train_weights, train_d)],
                )
                damage_clfs[key] = dmg_clf

            n_retrains += 1
            print(f"   retrain {n_retrains}  (data up to {retrain_time.date()})")

        split_time    = ref_ts.time_index[t0]
        pred_series_c = [ts.drop_after(split_time) for ts in full_target_c]

        preds_c = clf.predict(
            n=OUTPUT_CHUNK_LEN, series=pred_series_c,
            past_covariates=full_past_covs_c, future_covariates=full_fut_covs_c,
            predict_likelihood_parameters=True, show_warnings=False,
        )

        preds_d_all = {}
        for key in damage_classes:
            dc = damage_classes[key]
            full_past_covs_d = dc['get_covs_and_encodings']['FPC']
            full_fut_covs_d  = dc['get_covs_and_encodings']['FFC']
            pred_series_d = [ts.drop_after(split_time) for ts in full_target_d[key]]
            preds_d_all[key] = damage_clfs[key].predict(
                n=OUTPUT_CHUNK_LEN, series=pred_series_d,
                past_covariates=full_past_covs_d, future_covariates=full_fut_covs_d,
                predict_likelihood_parameters=True, show_warnings=False,
            )

        for r_idx, pred_c in enumerate(preds_c):
            prob_event_ts = pred_c.univariate_component(pred_c.n_components - 1)
            prob_event    = prob_event_ts.values().ravel()
            fold_preds_c[r_idx].append(prob_event_ts)

            for key in damage_classes:
                pred_d      = preds_d_all[key][r_idx]
                prob_dmg_ts = pred_d.univariate_component(pred_d.n_components - 1)
                prob_dmg    = prob_dmg_ts.values().ravel()
                hurdle_ts   = TimeSeries.from_times_and_values(
                    prob_event_ts.time_index, (prob_event * prob_dmg).reshape(-1, 1)
                )
                fold_preds_d[key][r_idx].append(prob_dmg_ts)
                fold_preds_h[key][r_idx].append(hurdle_ts)

        n_preds += 1

    print(f"   {n_preds} daily predictions, {n_retrains} retrains complete")
    return fold_preds_c, fold_preds_d, fold_preds_h, full_target_c, full_target_d


In [ ]:
print("Running final test evaluation (retrain every 7 days) ...")
test_preds_c, test_preds_d, test_preds_h, full_target_c, full_target_d = run_final_test()


This model will treat target `series` lagged values as numeric input features (and not categorical).


Running final test evaluation (retrain every 7 days) ...


This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).
This model will treat target `series` lagged values as numeric input features (and not categorical).


   retrain 1  (data up to 2024-08-05)


In [ ]:
# Collect test predictions
test_long_c = (
    collect_predictions_long(full_target_c, test_preds_c, region_names)
    .rename(columns={"y_pred": "y_prob"})
)

test_long_d = {}
test_long_h = {}
for key in damage_classes:
    test_long_d[key] = (
        collect_predictions_long(full_target_d[key], test_preds_d[key], region_names)
        .rename(columns={"y_pred": "y_prob"})
    )
    test_long_h[key] = (
        collect_predictions_long(full_target_d[key], test_preds_h[key], region_names)
        .rename(columns={"y_pred": "y_prob"})
    )

# Evaluate uncalibrated
test_results_c = evaluate_classif_long(test_long_c)
gc = test_results_c["global"]
print("=== Final Test — Event Classifier (uncalibrated) ===")
print(f"  F1      : {gc['F1']:.4f}")
print(f"  ROC-AUC : {gc['ROC_AUC']:.4f}")
print(f"  PR-AUC  : {gc['PR_AUC']:.4f}")
print(f"  Brier   : {gc['Brier']:.4f}")
print(f"  n       : {gc['n']:,}")

for key in damage_classes:
    label = key.replace("act_drone_infra_ua_", "").replace("_intent", "")
    gh = evaluate_classif_long(test_long_h[key])["global"]
    print(f"\n=== Final Test — [{label}] Hurdle (uncalibrated) ===")
    print(f"  F1      : {gh['F1']:.4f}  |  ROC-AUC : {gh['ROC_AUC']:.4f}  |  PR-AUC : {gh['PR_AUC']:.4f}  |  Brier : {gh['Brier']:.4f}")

print("\n=== Final Test — Event Classifier Per-horizon ===")
print(test_results_c["per_horizon"][["horizon", "F1", "ROC_AUC", "PR_AUC", "Brier", "n"]])


In [ ]:
# Apply CV-fitted calibrators to test predictions (no leakage)
test_long_c_reset = test_long_c.reset_index(drop=True)
test_long_c_cal   = test_long_c_reset.copy()
test_long_c_cal["y_prob"] = apply_calibrator_per_horizon(
    test_long_c_reset, calibrators_event, method=CAL_METHOD
)

test_long_d_cal = {}
test_long_h_cal = {}
for key in damage_classes:
    df_d_reset = test_long_d[key].reset_index(drop=True)
    test_long_d_cal[key]           = df_d_reset.copy()
    test_long_d_cal[key]["y_prob"] = apply_calibrator_per_horizon(
        df_d_reset, damage_calibrators[key], method=CAL_METHOD
    )
    df_h_reset = test_long_h[key].reset_index(drop=True).copy()
    df_h_reset["y_prob"] = test_long_c_cal["y_prob"].values * test_long_d_cal[key]["y_prob"].values
    test_long_h_cal[key] = df_h_reset

# Print calibrated results
gc_unc = test_results_c["global"]
gc_cal = evaluate_classif_long(test_long_c_cal)["global"]
print(f"=== Final Test — Event Classifier after {CAL_METHOD} calibration ===")
print(f"  Brier   : {gc_unc['Brier']:.4f}  ->  {gc_cal['Brier']:.4f}")
print(f"  ROC-AUC : {gc_unc['ROC_AUC']:.4f}  ->  {gc_cal['ROC_AUC']:.4f}")
print(f"  PR-AUC  : {gc_unc['PR_AUC']:.4f}  ->  {gc_cal['PR_AUC']:.4f}")

print("\n=== Calibrated Event Classifier — Per-horizon ===")
print(evaluate_classif_long(test_long_c_cal)["per_horizon"][["horizon", "F1", "ROC_AUC", "PR_AUC", "Brier", "n"]])

for key in damage_classes:
    label = key.replace("act_drone_infra_ua_", "").replace("_intent", "")
    gh_unc = evaluate_classif_long(test_long_h[key])["global"]
    gh_cal = evaluate_classif_long(test_long_h_cal[key])["global"]
    print(f"\n=== [{label}] Hurdle after {CAL_METHOD} calibration ===")
    print(f"  Brier   : {gh_unc['Brier']:.4f}  ->  {gh_cal['Brier']:.4f}")
    print(f"  ROC-AUC : {gh_unc['ROC_AUC']:.4f}  ->  {gh_cal['ROC_AUC']:.4f}")
    print(f"  PR-AUC  : {gh_unc['PR_AUC']:.4f}  ->  {gh_cal['PR_AUC']:.4f}")
    print(f"\n=== [{label}] Calibrated Hurdle — Per-horizon ===")
    print(evaluate_classif_long(test_long_h_cal[key])["per_horizon"][["horizon", "F1", "ROC_AUC", "PR_AUC", "Brier", "n"]])
